In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import BaseMessage

In [ ]:
from typing import List, Dict, Any, Literal
from typing_extensions import Annotated, TypedDict
from langgraph.graph import MessagesState
from langgraph.types import Command
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import SystemMessage, HumanMessage , AIMessage

from backend.chatbot.prompts.router.supervisor_prompt import prompt as supervisor_prompt

def merge_dicts(old, new): # i used this to let collected output gest updated before overwriting
    return {**old, **new}

class GraphState(MessagesState):
    remaining_agents: List[str]
    collected_outputs: Annotated[Dict[str, Any], merge_dicts]
    final_answer: str


class RouterState(TypedDict):
    next: str
    reason: str
    confidence: float

agents_description = """
        attachment: Detects attachment style patterns (secure/anxious/avoidant/disorganized) from relational signals.
        clinical_disorder: Detects possible clinical syndrome patterns from symptom clusters.
        cognetive_distortion: Detects distorted thinking patterns (catastrophizing, black-and-white thinking, etc.).
        functional_level: Assesses impairment in work, social, self-care, and daily functioning.
        personal_traits: Detects stable personality trait tendencies from repeated behavioral-emotional patterns.
        relational_pattern: Detects interpersonal dynamics, boundaries, dependency, conflict, and communication patterns.
        schema: Detects early maladaptive schemas and core beliefs.
""".strip()


def Make_router_node(llm: BaseChatModel, test_context: str):
    def router_node(state: GraphState) -> Command[
        Literal[
            "schema",
            "attachment",
            "clinical_disorder",
            "cognetive_distortion",
            "functional_level",
            "personal_traits",
            "relational_pattern",
            "meta_analist",
            "__end__",
        ]
    ]:
        remaining_agents = state.get("remaining_agents", [])
        if not remaining_agents:
            return Command(goto="meta_analist")

        options = remaining_agents 

        system_prompt = (
              supervisor_prompt
              + "\n\nAvailable agents: " + ", ".join(options)
              + "\n\nAgent descriptions:\n" + agents_description
            )
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=test_context),
        ]

        decision = llm.invoke(messages)
        return decision

    return router_node

In [ ]:
import os
from langchain_openai import ChatOpenAI
from user_context import first, second, third


llm = ChatOpenAI(
    model="deepseek-chat",
    api_key='sk-1666caf6a268456d8df5b5da9853d5d6',
    base_url="https://api.deepseek.com",
    temperature=0.7,
)


supervisor = Make_router_node(llm, test_context=third)

state = {
    "messages": [],  
    "remaining_agents": [
        "schema",
        "attachment",
        "clinical_disorder",
        "cognetive_distortion",
        "functional_level",
        "personal_traits",
        "relational_pattern",
    ],
    "collected_outputs": {},
}

# response = supervisor(state)
# print(response)

In [ ]:
# from pathlib import Path
# md_text = response.content

# out_path = Path("router_output.md")

# out_path.write_text(md_text, encoding="utf-8")

# print(f"Saved: {out_path.resolve()}")

In [ ]:
# import json

# data = json.loads(response.content)
# data

In [ ]:
# selected_agents = data['selected_agents']
# selected_agents

### agent list

In [ ]:

from backend.chatbot.prompts.attachment import prompt as attachment_prompt
from backend.chatbot.prompts.schema import prompt as  schema_prompt 
from backend.chatbot.prompts.clinical_disorder import prompt as clinical_disorder_prompt 
from backend.chatbot.prompts.personal_traits import prompt as personal_train_prompt 
from backend.chatbot.prompts.relational_pattern import prompt as relational_pattern_prompt
from backend.chatbot.prompts.functional_level import prompt as functional_level_prompt
from backend.chatbot.prompts.cognetive_distortion import prompt as cognitive_distortation_prompt
import json


def create_agent_node(llm  , prompt , node_name) :
    
    def agent(state:GraphState)-> Command[Literal['supervisor']]: 
        user_context = state["messages"][-1].content
        messages = [
            SystemMessage(content= prompt), 
            HumanMessage(content = user_context)
        ]

        response = llm.invoke(messages)
        # updated_messages = state['messages'] + [
        #     AIMessage(content = response.content , name = node_name)
        # ]
        data = json.loads(response.content)


        return Command(
            update = {
                  'collected_outputs': data 
            }, 
            goto = 'supervisor'
        )
    return agent

attachment_agent = create_agent_node(llm, attachment_prompt , node_name = 'atthchment')
schema_agent = create_agent_node(llm, schema_prompt , node_name = 'schema')
clinical_disorder_agent = create_agent_node(llm, clinical_disorder_prompt , node_name = 'clinical_disorder')
personal_trait_agent = create_agent_node(llm, personal_train_prompt , node_name = 'personal_trait')
relational_pattern_agent = create_agent_node(llm, relational_pattern_prompt , node_name = 'relational_pattern')
functional_level_agent = create_agent_node(llm, functional_level_prompt , node_name = 'functional_level')
cognitive_distortion_agent = create_agent_node(llm, cognitive_distortation_prompt , node_name = 'cognitive_distortion')

### planner and executor code 

In [ ]:
from typing import List, Dict, Any
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, END
from langgraph.types import Command
from langgraph.graph import MessagesState
from backend.chatbot.prompts.meta_analist import prompt as meta_analist_prompt
from user_context import first, second, third
import json

def router_node(state: GraphState) -> Command[
       Literal['supervisor']
    ]:
        remaining_agents = state.get("remaining_agents", [])
        if not remaining_agents:
            return Command(goto="meta_analist")

        options = remaining_agents 

        system_prompt = (
              supervisor_prompt
              + "\n\nAvailable agents: " + ", ".join(options)
              + "\n\nAgent descriptions:\n" + agents_description
            )
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=first),
        ]

        response = llm.invoke(messages)
        data = json.loads(response.content)

        return Command(
            goto = 'supervisor',
            update = {
                'remaining_agents' : data['selected_agents'],
                'collected_outputs': {} 
            }
        )


def supervisor_node(state: GraphState) -> Command[
    Literal[
        "attachment",
        "schema",
        "clinical_disorder",
        "personal_traits",
        "functional_level",
        "cognetive_distortion",
        "relational_pattern",
        "meta_analist", ]
        ]:
    remaining_agents = state.get("remaining_agents", [])

    if not remaining_agents:
        return Command(goto="meta_analist")

    next_agent = remaining_agents[0]

    return Command(
        goto=next_agent,
        update={
            "remaining_agents": remaining_agents[1:],  
        },
    )



def meta_analist_node(state: GraphState) -> Command:
    outputs = state.get("collected_outputs", {})
    messages = [
    SystemMessage(content=meta_analist_prompt),
    HumanMessage(content=json.dumps(outputs))
        ]
    response = llm.invoke(messages)
    result = json.loads(response.content)

    return Command(
        goto=END,
        update={"final_answer": result},
    )


builder = StateGraph(GraphState)

# Nodes
builder.add_node("router", router_node)
builder.add_node("supervisor", supervisor_node)



agent_info = {
    "attachment" : attachment_prompt,
    "schema" : schema_prompt,
    "clinical_disorder" : clinical_disorder_prompt,
    "personal_traits" : personal_train_prompt,
    "functional_level" : functional_level_prompt,
    'cognetive_distortion' : cognitive_distortation_prompt,
    'relational_pattern' : relational_pattern_prompt
}

for name , promt in agent_info.items():
    builder.add_node(name, create_agent_node(llm, promt , node_name = name))

builder.add_node("meta_analist", meta_analist_node)

# Entry point
builder.set_entry_point("router")

# Compile
graph = builder.compile()


# result = graph.invoke(
#     {
#         "messages": [],
#     }
# )

# from IPython.display import Image, display

# img = graph.get_graph().draw_mermaid_png()
# display(Image(img))


# display(Image(img))
print(graph.get_graph().draw_mermaid())

